# FinGuide BiLSTM — 7-Day Expense Predictor
**Trained on real MoMo transactions from `backend/finguide.db`**  
Users: **Michael** (user_id=1) and **Quercy** (user_id=2)

### What this notebook does
1. Pulls expense transactions from SQLite  
2. Aggregates to a daily time series per user  
3. Engineers temporal + contextual features  
4. Builds 30-day lookback → 7-day prediction sequences  
5. Trains a Bidirectional LSTM with dual input / triple output  
6. Saves all artifacts to `ML/models/` — exact names expected by `finguide_inference.py`

In [ ]:
# ── 1. Imports & reproducibility ───────────────────────────────────────────
import os, sqlite3, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow : {tf.__version__}')
print(f'NumPy      : {np.__version__}')
print(f'Pandas     : {pd.__version__}')

: 

In [ ]:
# ── 2. Configuration ────────────────────────────────────────────────────────
# Paths (relative to ML/ directory)
DB_PATH    = os.path.join('..', 'backend', 'finguide.db')
MODEL_DIR  = 'models'

# Training targets
TARGET_USERS = [1, 2]          # Michael (1) and Quercy (2)

# Sequence parameters
LOOKBACK   = 30                # days of history per input window
FORECAST   = 7                 # days to predict ahead

# Training hyper-parameters
BATCH_SIZE = 32
EPOCHS     = 150
LR         = 1e-3

# Domain knowledge: essential spending categories for Rwanda context
ESSENTIAL_CATS = {'UTILITIES', 'AIRTIME_DATA'}

# Days considered 'payday proximity' (start/mid/end of month)
PAYDAY_DAYS = {1, 2, 15, 28, 29, 30, 31}

print('Config OK')
print(f'DB path : {os.path.abspath(DB_PATH)}')
print(f'Model dir: {os.path.abspath(MODEL_DIR)}')

In [ ]:
# ── 3. Load data from SQLite ────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)

df_raw = pd.read_sql("""
    SELECT
        t.user_id,
        u.full_name,
        t.transaction_date,
        t.transaction_type,
        t.category,
        t.amount,
        t.balance_after
    FROM transactions t
    JOIN users u ON u.id = t.user_id
    WHERE t.user_id IN (1, 2)
    ORDER BY t.user_id, t.transaction_date
""", conn)
conn.close()

df_raw['transaction_date'] = pd.to_datetime(df_raw['transaction_date'])
df_raw['category']         = df_raw['category'].fillna('OTHER').str.upper()
df_raw['transaction_type'] = df_raw['transaction_type'].str.upper()

print(f'Total rows loaded : {len(df_raw):,}')
print()
print(df_raw.groupby(['full_name', 'transaction_type'])['amount'].agg(['count', 'sum']).round(0))

In [ ]:
# ── 4. Filter expenses & quick EDA ─────────────────────────────────────────
expenses = df_raw[df_raw['transaction_type'] == 'EXPENSE'].copy()
print(f'Expense transactions : {len(expenses):,}\n')

cat_summary = expenses.groupby('category')['amount'].agg(['count', 'sum', 'mean']).round(0)
cat_summary.columns = ['Count', 'Total RWF', 'Avg RWF']
print(cat_summary.sort_values('Total RWF', ascending=False))

# Quick date range per user
print()
for uid, name in [(1, 'Michael'), (2, 'Quercy')]:
    u = expenses[expenses['user_id'] == uid]
    print(f'{name:10s} | {len(u):4d} txns | {u.transaction_date.min().date()} → {u.transaction_date.max().date()}')

In [ ]:
# ── 5. Daily aggregation (per user) ────────────────────────────────────────
def make_daily_series(user_expense_df, user_id):
    """
    Resample expense transactions to one row per calendar day.
    amount   = sum of expenses on that day (0 on non-spending days)
    category = category with the highest single expense on that day
    """
    df = user_expense_df.copy()
    df['date'] = df['transaction_date'].dt.normalize()

    daily_amount = df.groupby('date')['amount'].sum()
    # Pick the category of the single largest transaction on each day
    daily_cat = (
        df.sort_values('amount', ascending=False)
          .groupby('date')['category']
          .first()
    )

    full_range = pd.date_range(
        start=daily_amount.index.min(),
        end=daily_amount.index.max(),
        freq='D'
    )

    daily = pd.DataFrame(index=full_range)
    daily['amount']   = daily_amount.reindex(full_range, fill_value=0.0)
    daily['category'] = daily_cat.reindex(full_range, fill_value='No_Transaction')
    daily['user_id']  = user_id
    return daily


user_dailies = []
for uid in TARGET_USERS:
    u_df  = expenses[expenses['user_id'] == uid]
    daily = make_daily_series(u_df, uid)
    user_dailies.append(daily)
    nonzero = (daily['amount'] > 0).sum()
    print(f'User {uid}: {len(daily)} days total  |  {nonzero} spending days  |  '
          f'{daily.index.min().date()} → {daily.index.max().date()}')

In [ ]:
# ── 6. Feature engineering ─────────────────────────────────────────────────
# These 7 temporal / contextual features + the amount will form the 8-feature
# numerical input that finguide_inference.py expects.

def add_features(daily_df):
    df = daily_df.copy()
    df['day_of_week']         = df.index.dayofweek            # 0=Mon … 6=Sun
    df['day_of_month']        = df.index.day
    df['month']               = df.index.month
    df['is_weekend']          = (df['day_of_week'] >= 5).astype(int)
    df['is_payday_proximity'] = df['day_of_month'].apply(
        lambda x: 1 if x in PAYDAY_DAYS else 0
    )
    df['is_essential']        = df['category'].apply(
        lambda x: 1 if x in ESSENTIAL_CATS else 0
    )
    # Rolling 7-day average spend as a proxy for liquidity buffer
    df['liquidity_buffer']    = df['amount'].rolling(7, min_periods=1).mean()
    return df


user_dailies = [add_features(d) for d in user_dailies]
print('Feature columns:', user_dailies[0].columns.tolist())

In [ ]:
# ── 7. Category encoding ────────────────────────────────────────────────────
# Collect every category that appears across both users
all_cats = set()
for d in user_dailies:
    all_cats.update(d['category'].unique())
all_cats = sorted(all_cats)

cat_encoder = LabelEncoder()
cat_encoder.fit(all_cats)

for i, d in enumerate(user_dailies):
    user_dailies[i]['cat_encoded'] = cat_encoder.transform(d['category'])

NUM_CATEGORIES = len(all_cats)
print(f'Total categories : {NUM_CATEGORIES}')
print('Categories       :', all_cats)

In [ ]:
# ── 8. Sequence building ────────────────────────────────────────────────────
# For each user independently (windows never cross user boundaries):
#   X_num  : (N, 30, 8)  — 8 numerical features per day
#   X_cat  : (N, 30)     — category index per day
#   y_amount : (N,)      — total expenses in the next 7 days (raw RWF)
#   y_cat    : (N,)      — dominant category index in the next 7 days
#   y_vol    : (N,)      — spending volatility [0, 1]

NUMERICAL_COLS = [
    'amount',              # will be log1p-scaled with amount_scaler
    'day_of_week',
    'day_of_month',
    'month',
    'is_weekend',
    'is_payday_proximity',
    'is_essential',
    'liquidity_buffer',
]  # 8 features — must match finguide_inference.py exactly


def build_sequences(daily_df):
    df  = daily_df.reset_index(drop=False).rename(columns={'index': 'date'})
    n   = len(df)

    X_num_list, X_cat_list = [], []
    y_amount_list, y_cat_list, y_vol_list = [], [], []

    for i in range(LOOKBACK, n - FORECAST + 1):
        window        = df.iloc[i - LOOKBACK : i]
        target_window = df.iloc[i : i + FORECAST]

        # ── inputs ──
        X_num_list.append(window[NUMERICAL_COLS].values.astype(float))
        X_cat_list.append(window['cat_encoded'].values)

        # ── targets ──
        target_amts = target_window['amount'].values

        # 1. 7-day total spend
        y_amount_list.append(target_amts.sum())

        # 2. Dominant category (by amount)
        if target_amts.sum() > 0:
            top_cat = (
                target_window.groupby('cat_encoded')['amount']
                             .sum()
                             .idxmax()
            )
        else:
            top_cat = cat_encoder.transform(['No_Transaction'])[0]
        y_cat_list.append(top_cat)

        # 3. Volatility: coefficient of variation, clipped [0, 1]
        mean_v = target_amts.mean()
        vol    = np.std(target_amts) / (mean_v + 1e-6) if mean_v > 0 else 0.0
        y_vol_list.append(min(vol, 1.0))

    return (
        np.array(X_num_list, dtype=np.float32),
        np.array(X_cat_list, dtype=np.int32),
        np.array(y_amount_list, dtype=np.float32),
        np.array(y_cat_list, dtype=np.int32),
        np.array(y_vol_list, dtype=np.float32),
    )


all_seqs = [build_sequences(d) for d in user_dailies]

X_num    = np.vstack([s[0] for s in all_seqs])     # (N, 30, 8)
X_cat    = np.vstack([s[1] for s in all_seqs])     # (N, 30)
y_amount = np.concatenate([s[2] for s in all_seqs])  # (N,)
y_cat    = np.concatenate([s[3] for s in all_seqs])  # (N,)
y_vol    = np.concatenate([s[4] for s in all_seqs])  # (N,)

print(f'Total sequences  : {len(X_num):,}')
print(f'X_num  shape     : {X_num.shape}')
print(f'X_cat  shape     : {X_cat.shape}')
print(f'y_amount stats   : mean={y_amount.mean():,.0f}  max={y_amount.max():,.0f}  zeros={( y_amount == 0).sum()}')

In [ ]:
# ── 9. Preprocessing ────────────────────────────────────────────────────────

# ── 9a. Log-transform amounts (column 0) in the input windows ──
X_num_proc = X_num.copy()
X_num_proc[:, :, 0] = np.log1p(X_num_proc[:, :, 0])   # daily amounts → log space
y_amount_log        = np.log1p(y_amount)               # 7-day targets → log space

# ── 9b. amount_scaler — fit on the UNION of daily amounts + 7-day targets ──
# This ensures inverse_transform in inference is in the same distribution.
combined_log_amounts = np.concatenate([
    X_num_proc[:, :, 0].flatten(),
    y_amount_log,
])
amount_scaler = StandardScaler()
amount_scaler.fit(combined_log_amounts.reshape(-1, 1))

# Scale input daily amounts
flat_input_amounts          = X_num_proc[:, :, 0].reshape(-1, 1)
X_num_proc[:, :, 0]         = amount_scaler.transform(flat_input_amounts).reshape(len(X_num), LOOKBACK)

# Scale 7-day targets
y_amount_scaled = amount_scaler.transform(y_amount_log.reshape(-1, 1)).flatten()

# ── 9c. feature_scaler — fit on columns 1-7 (temporal/contextual) ──
# Use raw (pre-log) values since these are already on sensible scales.
other_feats_raw    = X_num[:, :, 1:].reshape(-1, 7)   # (N*30, 7)
feature_scaler     = StandardScaler()
feature_scaler.fit(other_feats_raw)

X_num_proc[:, :, 1:] = feature_scaler.transform(
    X_num[:, :, 1:].reshape(-1, 7)
).reshape(len(X_num), LOOKBACK, 7)

X_num_final = X_num_proc   # shape (N, 30, 8)

# ── 9d. One-hot encode category targets ──
y_cat_onehot = tf.keras.utils.to_categorical(y_cat, num_classes=NUM_CATEGORIES).astype(np.float32)

# ── 9e. Volatility stays as-is (already [0,1]) ──
y_vol_scaled = y_vol.reshape(-1, 1)

# ── 9f. Train / validation / test split (80 / 10 / 10 chronological) ──
n        = len(X_num_final)
s1, s2   = int(n * 0.80), int(n * 0.90)

X_tr_n,  X_va_n,  X_te_n  = X_num_final[:s1],    X_num_final[s1:s2],   X_num_final[s2:]
X_tr_c,  X_va_c,  X_te_c  = X_cat[:s1],          X_cat[s1:s2],         X_cat[s2:]
y_tr_a,  y_va_a,  y_te_a  = y_amount_scaled[:s1], y_amount_scaled[s1:s2], y_amount_scaled[s2:]
y_tr_c,  y_va_c,  y_te_c  = y_cat_onehot[:s1],   y_cat_onehot[s1:s2],  y_cat_onehot[s2:]
y_tr_v,  y_va_v,  y_te_v  = y_vol_scaled[:s1],   y_vol_scaled[s1:s2],  y_vol_scaled[s2:]

print(f'Train : {len(X_tr_n):4d}   Val : {len(X_va_n):4d}   Test : {len(X_te_n):4d}')

In [ ]:
# ── 10. Model definition ────────────────────────────────────────────────────
# Architecture mirrors what finguide_inference.py expects:
#   Input 1 : (batch, 30, 8)  numerical features
#   Input 2 : (batch, 30)     category indices  → embedding → (batch, 30, 8)
#   Output 1: (batch, 1)      scaled 7-day total expense
#   Output 2: (batch, C)      category softmax
#   Output 3: (batch, 1)      volatility score [0,1]

from tensorflow.keras import regularizers

def build_bilstm(lookback, num_features, num_categories):
    # ── Inputs ──
    inp_num = keras.Input(shape=(lookback, num_features), name='numerical_input')
    inp_cat = keras.Input(shape=(lookback,),               name='category_input')

    # ── Category embedding ──
    cat_embed = layers.Embedding(
        input_dim=num_categories + 1,
        output_dim=4, # Reduced embedding dimension
        name='category_embedding'
    )(inp_cat)

    # ── Merge ──
    merged = layers.Concatenate(axis=-1, name='merge')([inp_num, cat_embed])

    # ── BiLSTM stack (Reduced capacity to prevent overfitting on ~1.7k samples) ──
    x = layers.Bidirectional(layers.LSTM(32, return_sequences=False, 
                                       kernel_regularizer=regularizers.l2(1e-4)), name='bilstm_1')(merged)
    x = layers.Dropout(0.3)(x)

    # ── Shared dense representation ──
    shared = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4), name='shared_1')(x)
    shared = layers.Dropout(0.2)(shared)

    # ── Outputs ──
    out_amount     = layers.Dense(1,             name='amount_output')(shared)
    out_category   = layers.Dense(num_categories, activation='softmax', name='category_output')(shared)
    out_volatility = layers.Dense(1,             activation='sigmoid',  name='volatility_output')(shared)

    model = keras.Model(
        inputs  =[inp_num, inp_cat],
        outputs =[out_amount, out_category, out_volatility]
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss={
            'amount_output'    : 'huber', # More robust to outliers than MSE
            'category_output'  : 'categorical_crossentropy',
            'volatility_output': 'mse',
        },
        loss_weights={
            'amount_output'    : 1.0,
            'category_output'  : 0.2,
            'volatility_output': 0.1,
        },
        metrics={
            'amount_output'  : ['mae'],
            'category_output': ['accuracy'],
        }
    )
    return model


model = build_bilstm(LOOKBACK, len(NUMERICAL_COLS), NUM_CATEGORIES)
model.summary()

In [ ]:
# ── 11. Training ────────────────────────────────────────────────────────────
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(MODEL_DIR, 'bilstm_7day_best.h5'),
        monitor='val_amount_output_mae',
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=20,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=8,
        min_lr=1e-5,
        verbose=1,
    ),
]

history = model.fit(
    [X_tr_n, X_tr_c],
    [y_tr_a, y_tr_c, y_tr_v],
    validation_data=(
        [X_va_n, X_va_c],
        [y_va_a, y_va_c, y_va_v],
    ),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# ── 12. Evaluation ──────────────────────────────────────────────────────────
# Reload the best checkpoint
best_model = keras.models.load_model(os.path.join(MODEL_DIR, 'bilstm_7day_best.h5'))

preds           = best_model.predict([X_te_n, X_te_c], verbose=0)
pred_amt_scaled = preds[0].flatten()
pred_cat_probs  = preds[1]
pred_vol        = preds[2].flatten()

# Inverse-transform amounts
pred_log  = amount_scaler.inverse_transform(pred_amt_scaled.reshape(-1, 1)).flatten()
pred_rwf  = np.expm1(pred_log)

true_log  = amount_scaler.inverse_transform(y_te_a.reshape(-1, 1)).flatten()
true_rwf  = np.expm1(true_log)

# Metrics
rmse     = np.sqrt(mean_squared_error(true_rwf, pred_rwf))
mae      = mean_absolute_error(true_rwf, pred_rwf)
r2       = r2_score(true_rwf, pred_rwf)
cat_acc  = accuracy_score(
    np.argmax(y_te_c, axis=1),
    np.argmax(pred_cat_probs, axis=1)
)

print('=' * 45)
print(' TEST SET METRICS')
print('=' * 45)
print(f'  RMSE  : {rmse:>12,.0f} RWF')
print(f'  MAE   : {mae:>12,.0f} RWF')
print(f'  R²    : {r2:>12.4f}')
print(f'  Cat.  : {cat_acc:>12.1%}')
print('=' * 45)

# ── Plots ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training curves
axes[0].plot(history.history['loss'],     label='Train loss')
axes[0].plot(history.history['val_loss'], label='Val loss')
axes[0].set_title('Loss during training')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# Predicted vs actual (first 60 test samples)
n_show = min(60, len(true_rwf))
axes[1].plot(true_rwf[:n_show], label='Actual',    marker='o', ms=4, alpha=0.7)
axes[1].plot(pred_rwf[:n_show], label='Predicted', marker='x', ms=4, alpha=0.7)
axes[1].set_title('7-Day Expense: Predicted vs Actual')
axes[1].set_xlabel('Test sample')
axes[1].set_ylabel('RWF')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'bilstm_training_history.png'), dpi=120)
plt.show()
print('Saved training_history.png')

In [ ]:
# ── 13. Save all artifacts ──────────────────────────────────────────────────
# These filenames are hardcoded in backend/app/core/finguide_inference.py — do not rename.

joblib.dump(amount_scaler,  os.path.join(MODEL_DIR, 'bilstm_7day_target_scaler.joblib'))
joblib.dump(feature_scaler, os.path.join(MODEL_DIR, 'bilstm_7day_feature_scaler.joblib'))
joblib.dump(cat_encoder,    os.path.join(MODEL_DIR, 'bilstm_7day_category_encoder.joblib'))

metadata = {
    'model_name'        : 'FinGuide BiLSTM 7-Day Aggregate Predictor',
    'version'           : '4.0',
    'trained_date'      : datetime.now().isoformat(),
    'target'            : 'Rolling 7-day expense sum (RWF)',
    'lookback_days'     : LOOKBACK,
    'forecast_horizon'  : FORECAST,
    'input_features'    : NUMERICAL_COLS,
    'num_categories'    : NUM_CATEGORIES,
    'categories'        : list(cat_encoder.classes_),
    'essential_cats'    : list(ESSENTIAL_CATS),
    'training_users'    : TARGET_USERS,
    'train_samples'     : len(X_tr_n),
    'val_samples'       : len(X_va_n),
    'test_samples'      : len(X_te_n),
    'metrics'           : {
        'rmse'              : float(rmse),
        'mae'               : float(mae),
        'r2'                : float(r2),
        'category_accuracy' : float(cat_acc),
    },
    'preprocessing'     : {
        'amount_input'  : 'log1p → same StandardScaler as target',
        'features_1_7'  : 'StandardScaler (raw values)',
        'target'        : 'log1p(7-day total) → StandardScaler',
    },
}
joblib.dump(metadata, os.path.join(MODEL_DIR, 'bilstm_7day_metadata.joblib'))

print('Artifacts saved:')
for fname in [
    'bilstm_7day_best.h5',
    'bilstm_7day_target_scaler.joblib',
    'bilstm_7day_feature_scaler.joblib',
    'bilstm_7day_category_encoder.joblib',
    'bilstm_7day_metadata.joblib',
]:
    path   = os.path.join(MODEL_DIR, fname)
    status = 'OK  ' if os.path.exists(path) else 'MISSING'
    size   = os.path.getsize(path) if os.path.exists(path) else 0
    print(f'  [{status}] {fname:45s}  {size/1024:6.1f} KB')

In [ ]:
# ── 14. Smoke test — simulate one inference call ────────────────────────────
# Replicates exactly what finguide_inference.py does at prediction time.
# Uses the last 30 days of Quercy's data as a sample.

loaded_model    = keras.models.load_model(os.path.join(MODEL_DIR, 'bilstm_7day_best.h5'))
loaded_a_scaler = joblib.load(os.path.join(MODEL_DIR, 'bilstm_7day_target_scaler.joblib'))
loaded_f_scaler = joblib.load(os.path.join(MODEL_DIR, 'bilstm_7day_feature_scaler.joblib'))
loaded_encoder  = joblib.load(os.path.join(MODEL_DIR, 'bilstm_7day_category_encoder.joblib'))

# Grab last 30 rows from Quercy's daily series (user_dailies[1])
sample = user_dailies[1].tail(30)

# Build a minimal transaction list as the API would send
sample_transactions = [
    {'date': str(row.Index.date()), 'amount': row.amount,
     'category': row.category, 'type': 'EXPENSE' if row.amount > 0 else 'None'}
    for row in sample.itertuples()
]

# ------- replicate _preprocess_user_data logic -------
df = pd.DataFrame(sample_transactions)
df['Date']   = pd.to_datetime(df['date'])
df['Amount'] = df['amount'].astype(float)
df = df.set_index('Date').sort_index()

end_date   = df.index.max()
start_date = end_date - pd.Timedelta(days=29)
full_range = pd.date_range(start=start_date, end=end_date, freq='D')

df_daily = df.reindex(full_range)
df_daily['Amount']   = df_daily['Amount'].fillna(0)
df_daily['category'] = df_daily['category'].fillna('No_Transaction').str.upper()

df_daily['Day_of_Week']         = df_daily.index.dayofweek
df_daily['Day_of_Month']        = df_daily.index.day
df_daily['Month']               = df_daily.index.month
df_daily['Is_Weekend']          = (df_daily['Day_of_Week'] >= 5).astype(int)
df_daily['Is_Payday_Proximity'] = df_daily['Day_of_Month'].apply(
    lambda x: 1 if x in PAYDAY_DAYS else 0
)
df_daily['Is_Essential']        = df_daily['category'].apply(
    lambda x: 1 if x in ESSENTIAL_CATS else 0
)
df_daily['Liquidity_Buffer']    = df_daily['Amount'].rolling(7, min_periods=1).mean()

known_cats = set(loaded_encoder.classes_)
df_daily['cat_safe']    = df_daily['category'].apply(
    lambda x: x if x in known_cats else 'No_Transaction'
)
df_daily['Cat_Encoded'] = loaded_encoder.transform(df_daily['cat_safe'])

df_daily['Amount_Log'] = np.log1p(df_daily['Amount'])
amount_scaled_in       = loaded_a_scaler.transform(df_daily[['Amount_Log']])

other_feats_in   = df_daily[['Day_of_Week', 'Day_of_Month', 'Month',
                              'Is_Weekend', 'Is_Payday_Proximity',
                              'Is_Essential', 'Liquidity_Buffer']].values
features_scaled_in = loaded_f_scaler.transform(other_feats_in)

X_num_smoke = np.hstack([amount_scaled_in, features_scaled_in]).reshape(1, 30, 8)
X_cat_smoke = df_daily['Cat_Encoded'].values.reshape(1, 30)

# Run prediction
smoke_preds    = loaded_model.predict([X_num_smoke, X_cat_smoke], verbose=0)
raw_amount     = loaded_a_scaler.inverse_transform([[smoke_preds[0][0][0]]])[0][0]
pred_amount_rw = np.expm1(raw_amount)
pred_cat_name  = loaded_encoder.inverse_transform([np.argmax(smoke_preds[1][0])])[0]
pred_conf      = round(float(1 - smoke_preds[2][0][0]) * 100, 1)

print('Smoke test passed!')
print(f'  Predicted 7-day expense : {pred_amount_rw:,.0f} RWF')
print(f'  Top category            : {pred_cat_name}')
print(f'  Confidence              : {pred_conf}%')